# Camera Discovery Live Test

Simplified 3-stage pipeline: `TargetResolver → CandidateDiscoveryEngine → ReviewAndValidationPipeline`.

LLMs are used as advisory evidence interpreters/rankers for target intent, geocoder candidate ranking, and candidate semantic review. Deterministic code/tools remain responsible for geometry verification, stream validation, trusted-output authorization, and final artifact writing.

This notebook supports single-location and multi-location queries, for example:

```text
Get me all cameras from Greenville, Texas
Get me all cameras from London, England and New York, New York
```


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = Path.cwd()
if (ROOT / 'src').exists(): sys.path.insert(0, str(ROOT / 'src'))
print('Root:', ROOT)

In [ ]:
RUN_PROFILE = os.environ.get('CAMERA_DISCOVERY_PROFILE', 'fast')
# User-controlled query. Edit this line or set CAMERA_DISCOVERY_QUERY before running.
# Camera terms such as "traffic cameras" are camera intent, not geography.
USER_QUERY = os.environ.get('CAMERA_DISCOVERY_QUERY', 'Get me all traffic cameras from California')
# Example multi-location query:
# USER_QUERY = 'Get me all cameras from London, England and New York, New York'
DISCOVERY_MODE = os.environ.get('CAMERA_DISCOVERY_DISCOVERY_MODE', 'both')
SOURCES_FILE = os.environ.get('CAMERA_DISCOVERY_SOURCES_FILE', 'SOURCES.md')
OUTPUT_DIR = Path(os.environ.get('CAMERA_DISCOVERY_OUTPUT_DIR', 'runs/notebook-live-test'))

# Candidate coordinate enrichment from real source data and optional geocoder calls.
os.environ.setdefault('CAMERA_DISCOVERY_ENABLE_CANDIDATE_GEOCODING', 'true')
os.environ.setdefault('CAMERA_DISCOVERY_MAX_CANDIDATE_GEOCODES', '25')

print('Profile:', RUN_PROFILE)
print('Query:', USER_QUERY)
print('Discovery mode:', DISCOVERY_MODE)
print('Sources file:', SOURCES_FILE)
print('Output dir:', OUTPUT_DIR)
print('Candidate geocoding:', os.environ.get('CAMERA_DISCOVERY_ENABLE_CANDIDATE_GEOCODING'))


In [ ]:
cmd = [
    sys.executable, '-m', 'camera_discovery.cli', 'run', USER_QUERY,
    '--profile', RUN_PROFILE,
    '--output-dir', str(OUTPUT_DIR),
    '--discovery-mode', DISCOVERY_MODE,
    '--sources-file', str(SOURCES_FILE),
]
for url in SEED_URLS:
    cmd.extend(['--seed-url', url])
print('$', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
print('exit:', result.returncode)
if result.returncode != 0: raise RuntimeError('camera-discovery run failed')


In [ ]:
from pathlib import Path
import json

for rel in ['logs/source_policy_summary.json', 'logs/candidate_discovery_summary.json', 'logs/run_summary.json']:
    path = OUTPUT_DIR / rel
    print('---', rel, 'exists=', path.exists())
    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2)[:4000])


In [ ]:
summary_path = OUTPUT_DIR / 'logs' / 'run_summary.json'
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
targets = summary.get('targets', [])
print('targets:', len(targets))
for t in targets:
    print(json.dumps({
        'target_id': t.get('target_id'),
        'target_label': t.get('target_label'),
        'canonical_target': t.get('canonical_target'),
        'geometry_status': t.get('geometry_status'),
        'bbox_verified': t.get('bbox_verified'),
        'trust_policy': t.get('trust_policy'),
    }, indent=2))
print(json.dumps({
    'unique_candidates': len(summary.get('candidates',{}).get('unique', [])),
    'coordinate_bearing': len(summary.get('candidates',{}).get('coordinate_bearing', [])),
    'trusted_geojson_features': summary.get('outputs',{}).get('trusted_geojson_features_written'),
    'untrusted_geojson_features': summary.get('outputs',{}).get('untrusted_geojson_features_written'),
}, indent=2))


In [ ]:
for rel in [
    'camera.geojson',
    'untrusted_camera_candidates.geojson',
    'map.html',
    'review_artifacts.zip',
    'logs/target_resolution_all.json',
    'logs/target_intent.json',
    'logs/geocoder_referee.json',
    'logs/candidate_semantic_review.json',
    'logs/geocoder_candidate_scores.json',
    'logs/output_summary.json',
]:
    p = OUTPUT_DIR / rel
    print(rel, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)

# Per-target diagnostics are written below logs/targets/<target_id>/ and candidates/<target_id>/.
for folder in sorted((OUTPUT_DIR / 'logs' / 'targets').glob('*')) if (OUTPUT_DIR / 'logs' / 'targets').exists() else []:
    print('target diagnostics:', folder.relative_to(OUTPUT_DIR))


## Camera candidate table

This cell first uses `camera_candidates_table.csv`, which is written from all non-rejected candidates, including rows without coordinates. If that table is unavailable, it falls back to `camera.geojson` or `untrusted_camera_candidates.geojson`. This means the notebook can still show candidate URLs even when no GeoJSON/map can be created.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from camera_discovery.utils.geojson_viewer import (
    load_camera_rows,
    select_camera_geojson,
    write_camera_table_csv,
)

TABLE_PATH = OUTPUT_DIR / 'camera_candidates_table.csv'
GEOJSON_PATH = select_camera_geojson(OUTPUT_DIR)

if TABLE_PATH.exists():
    df = pd.read_csv(TABLE_PATH)
    print(f'Loaded candidate table: {TABLE_PATH} rows={len(df)}')
    display(df)
elif GEOJSON_PATH:
    rows = load_camera_rows(GEOJSON_PATH)
    print(f'Loaded GeoJSON table source: {GEOJSON_PATH} rows={len(rows)}')
    if rows:
        df = pd.DataFrame(rows)
        display(df)
        TABLE_PATH = write_camera_table_csv(OUTPUT_DIR, rows)
        print('Wrote table:', TABLE_PATH)
    else:
        print('GeoJSON exists but contains zero features.')
else:
    print('No candidate table or GeoJSON found. Check candidates/*.jsonl and logs/candidate_discovery_summary.json.')


## Interactive camera map

The map loads `camera.geojson` first and falls back to `untrusted_camera_candidates.geojson`. If no coordinate-bearing candidates exist, the map cell will create an empty-state map while the table above still shows non-GeoJSON candidates.

In [ ]:
from IPython.display import HTML, display
from camera_discovery.utils.geojson_viewer import write_embedded_camera_map

MAP_PATH = write_embedded_camera_map(OUTPUT_DIR, GEOJSON_PATH, output_name='notebook_camera_map.html')
print('Notebook map:', MAP_PATH)
display(HTML(MAP_PATH.read_text(encoding='utf-8')))
